In [1]:
%pip install networkx matplotlib pydot

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%load_ext autoreload
%autoreload 2

import cpg.cpg_trie_parser as cpg_parser

In [3]:
from pathlib import Path
import pydot
import networkx as nx

SERVICE_NAME = "frontend"
dot_path = Path(f"services/{SERVICE_NAME}/export.dot")

dot_text = dot_path.read_text(encoding="utf-8")
graphs = pydot.graph_from_dot_data(dot_text)

P = graphs[0]
P

In [4]:
G = nx.MultiDiGraph()

for node in P.get_nodes():
    name = node.get_name()
    if name in (None, "node", "graph", "edge"):
        continue
    nid = str(name).strip('"')
    attrs = {k: v for k, v in node.get_attributes().items()}
    G.add_node(nid, **attrs)

for edge in P.get_edges():
    src = str(edge.get_source()).strip('"')
    dst = str(edge.get_destination()).strip('"')
    attrs = {k: v for k, v in edge.get_attributes().items()}
    G.add_edge(src, dst, **attrs)

print(G.number_of_nodes(), G.number_of_edges())

30548 164385


In [5]:
templates = cpg_parser.build_templates_from_cpg(G, max_ddg_depth=5)
root = cpg_parser.build_trie(templates)
cpg_parser.visualize_trie_matplotlib(root, output_path=f"output/{SERVICE_NAME}/trie.png")

[Trie] Saved matplotlib PNG -> output/frontend/trie.png


In [6]:
print("Templates:", len(templates))
for t in templates[:10]:
    print(f"[{t.call_node_id}] {t.method_name}() -> {t.raw_template} (static={t.static_count})")

Templates: 77
[30064771307] loadDeploymentDetails() -> failed to fetch the hostname for the pod <*> (static=8)
[30064771312] loadDeploymentDetails() -> failed to fetch the name of the cluster in which the pod is running <*> (static=14)
[30064771317] loadDeploymentDetails() -> failed to fetch the zone of the node where the pod is scheduled <*> (static=13)
[30064771328] loadDeploymentDetails() -> loaded deployment details <*> (static=3)
[30064772872] AddItem() -> method additem not implemented <*> (static=4)
[30064772874] GetCart() -> method getcart not implemented <*> (static=4)
[30064772876] EmptyCart() -> method emptycart not implemented <*> (static=4)
[30064772938] ListRecommendations() -> method listrecommendations not implemented <*> (static=4)
[30064772994] ListProducts() -> method listproducts not implemented <*> (static=4)
[30064772996] GetProduct() -> method getproduct not implemented <*> (static=4)


In [7]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# Step 1 - Collect all endpoints
from cpg.entrypoint import EntrypointDetector, Entrypoint

print("STEP 1: COLLECTING ALL ENDPOINTS")
print("-" * 40)
detector = EntrypointDetector(G)
all_entrypoints = detector.detect()
print(f"Found {len(all_entrypoints)} entrypoints:")
for i, ep in enumerate(all_entrypoints):
    print(f"  {i+1}. {ep.name} -> {ep.full_name}")

STEP 1: COLLECTING ALL ENDPOINTS
----------------------------------------
Found 62 entrypoints:
  1. init -> main.init
  2. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.CartItem.Descriptor
  3. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.AddItemRequest.Descriptor
  4. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.EmptyCartRequest.Descriptor
  5. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.GetCartRequest.Descriptor
  6. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.Cart.Descriptor
  7. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.Empty.Descriptor
  8. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.ListRecommendationsRequest.Descriptor
  9. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/

In [10]:
from cpg.log_flow import StaticLogFSM, START, RETURN, INCOMPLETE
from typing import Dict, Tuple
def visualize_log_fsm(
    fsm: StaticLogFSM,
    output_path: str = "log_fsm.png",
    max_label_length: int = 42,
    figsize: Tuple[float, float] | None = None,
) -> None:
    """
    Сохраняет StaticLogFSM в PNG.

    Требует:
        pip install matplotlib networkx

    Example:
        fsm = LogFlowExtractor(templates).extract(flow_result)
        visualize_log_fsm(fsm, "order_flow_fsm.png")
    """
    import textwrap

    import matplotlib.pyplot as plt
    import networkx as nx
    from matplotlib.patches import FancyBboxPatch

    graph = nx.DiGraph()

    graph.add_node(
        START,
        kind="start",
        label="START",
    )

    for state_id, point in fsm.states.items():
        method = _short_method_name(point.method_fullname)
        template = _shorten_label(point.template, max_label_length)

        graph.add_node(
            state_id,
            kind="logpoint",
            label=f"{method}\n{template}",
        )

    for terminal in sorted(fsm.terminals):
        if terminal == RETURN:
            graph.add_node(
                RETURN,
                kind="return",
                label="RETURN",
            )
        elif terminal == INCOMPLETE:
            graph.add_node(
                INCOMPLETE,
                kind="incomplete",
                label="INCOMPLETE",
            )

    for edge in fsm.edges:
        if edge.source not in graph:
            graph.add_node(
                edge.source,
                kind="unknown",
                label=edge.source,
            )

        if edge.target not in graph:
            graph.add_node(
                edge.target,
                kind="unknown",
                label=edge.target,
            )

        conditions = ", ".join(edge.conditions)

        graph.add_edge(
            edge.source,
            edge.target,
            label=conditions,
            partial=edge.partial,
            terminal=edge.is_terminal,
        )

    if not graph.nodes:
        raise ValueError("FSM is empty; nothing to visualize.")

    if figsize is None:
        node_count = max(1, len(graph.nodes))
        figsize = (
            min(28.0, max(13.0, node_count * 2.2)),
            min(20.0, max(8.0, node_count * 1.35)),
        )

    positions = _fsm_layout(graph)

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor("#ffffff")
    ax.set_facecolor("#f8fafc")
    ax.axis("off")

    ax.set_title(
        f"Static Log FSM\n{fsm.entrypoint_fullname}",
        fontsize=16,
        fontweight="bold",
        color="#0f172a",
        pad=22,
    )

    _draw_fsm_edges(
        ax=ax,
        graph=graph,
        positions=positions,
    )

    _draw_fsm_nodes(
        ax=ax,
        graph=graph,
        positions=positions,
    )

    _draw_fsm_legend(ax)

    xs = [x for x, _ in positions.values()]
    ys = [y for _, y in positions.values()]

    padding_x = max(1.5, (max(xs) - min(xs)) * 0.12)
    padding_y = max(1.5, (max(ys) - min(ys)) * 0.16)

    ax.set_xlim(min(xs) - padding_x, max(xs) + padding_x)
    ax.set_ylim(min(ys) - padding_y - 0.5, max(ys) + padding_y)

    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )
    plt.close(fig)

    print(f"Static Log FSM saved: {output_path}")


def _fsm_layout(graph) -> Dict[str, Tuple[float, float]]:
    """
    Сначала пробует Graphviz dot: он лучше читает ориентированные автоматы.
    Если pygraphviz/pydot не установлены — использует deterministic layout.
    """
    import networkx as nx

    try:
        from networkx.drawing.nx_pydot import graphviz_layout

        raw = graphviz_layout(graph, prog="dot")

        return {
            node_id: (float(x), -float(y))
            for node_id, (x, y) in raw.items()
        }
    except Exception:
        pass

    try:
        levels = _compute_fsm_levels(graph)
        level_groups: Dict[int, List[str]] = {}

        for node_id, level in levels.items():
            level_groups.setdefault(level, []).append(node_id)

        positions: Dict[str, Tuple[float, float]] = {}

        for level, node_ids in sorted(level_groups.items()):
            ordered_nodes = sorted(
                node_ids,
                key=lambda node_id: (
                    _node_kind_priority(graph.nodes[node_id].get("kind", "")),
                    graph.nodes[node_id].get("label", ""),
                    node_id,
                ),
            )

            width = len(ordered_nodes)
            for index, node_id in enumerate(ordered_nodes):
                x = index - (width - 1) / 2
                y = -level * 2.8
                positions[node_id] = (x * 4.3, y)

        return positions
    except Exception:
        positions = nx.spring_layout(
            graph,
            seed=42,
            k=2.5,
            iterations=100,
        )

        return {
            node_id: (float(x) * 10.0, float(y) * 10.0)
            for node_id, (x, y) in positions.items()
        }


def _compute_fsm_levels(graph) -> Dict[str, int]:
    """
    Строит уровни от START.

    Для циклов и back-edge используется кратчайшее расстояние от START.
    Узлы, недостижимые из START, размещаются после достижимых уровней.
    """
    import networkx as nx

    levels: Dict[str, int] = {}

    if START in graph:
        levels.update(nx.single_source_shortest_path_length(graph, START))

    max_level = max(levels.values(), default=0)
    unreachable = sorted(set(graph.nodes) - set(levels))

    for offset, node_id in enumerate(unreachable, start=1):
        levels[node_id] = max_level + offset

    return levels


def _node_kind_priority(kind: str) -> int:
    priority = {
        "start": 0,
        "logpoint": 1,
        "return": 2,
        "incomplete": 3,
        "unknown": 4,
    }
    return priority.get(kind, 99)


def _draw_fsm_edges(
    ax,
    graph,
    positions: Dict[str, Tuple[float, float]],
) -> None:
    import matplotlib.pyplot as plt
    import networkx as nx

    normal_edges = []
    partial_edges = []
    self_loops = []

    for source, target, data in graph.edges(data=True):
        if source == target:
            self_loops.append((source, target))
        elif data.get("partial"):
            partial_edges.append((source, target))
        else:
            normal_edges.append((source, target))

    nx.draw_networkx_edges(
        graph,
        positions,
        ax=ax,
        edgelist=normal_edges,
        arrows=True,
        arrowstyle="-|>",
        arrowsize=18,
        width=1.8,
        edge_color="#64748b",
        connectionstyle="arc3,rad=0.05",
        min_source_margin=26,
        min_target_margin=26,
    )

    nx.draw_networkx_edges(
        graph,
        positions,
        ax=ax,
        edgelist=partial_edges,
        arrows=True,
        arrowstyle="-|>",
        arrowsize=18,
        width=1.8,
        style="dashed",
        edge_color="#f59e0b",
        connectionstyle="arc3,rad=0.05",
        min_source_margin=26,
        min_target_margin=26,
    )

    nx.draw_networkx_edges(
        graph,
        positions,
        ax=ax,
        edgelist=self_loops,
        arrows=True,
        arrowstyle="-|>",
        arrowsize=18,
        width=1.8,
        edge_color="#7c3aed",
        connectionstyle="arc3,rad=0.36",
        min_source_margin=28,
        min_target_margin=28,
    )

    edge_labels = {
        (source, target): data["label"]
        for source, target, data in graph.edges(data=True)
        if data.get("label")
    }

    if edge_labels:
        nx.draw_networkx_edge_labels(
            graph,
            positions,
            edge_labels=edge_labels,
            ax=ax,
            font_size=8,
            font_color="#334155",
            rotate=False,
            label_pos=0.5,
            bbox={
                "boxstyle": "round,pad=0.22",
                "facecolor": "#ffffff",
                "edgecolor": "#cbd5e1",
                "alpha": 0.95,
            },
        )


def _draw_fsm_nodes(
    ax,
    graph,
    positions: Dict[str, Tuple[float, float]],
) -> None:
    from matplotlib.patches import FancyBboxPatch

    colors = {
        "start": {
            "face": "#059669",
            "edge": "#064e3b",
            "text": "#ffffff",
        },
        "logpoint": {
            "face": "#2563eb",
            "edge": "#1e3a8a",
            "text": "#ffffff",
        },
        "return": {
            "face": "#166534",
            "edge": "#14532d",
            "text": "#ffffff",
        },
        "incomplete": {
            "face": "#f59e0b",
            "edge": "#92400e",
            "text": "#1f2937",
        },
        "unknown": {
            "face": "#94a3b8",
            "edge": "#475569",
            "text": "#0f172a",
        },
    }

    for node_id, attributes in graph.nodes(data=True):
        x, y = positions[node_id]

        kind = attributes.get("kind", "unknown")
        label = attributes.get("label", node_id)
        palette = colors.get(kind, colors["unknown"])

        lines = label.splitlines()
        widest_line = max((len(line) for line in lines), default=8)

        width = max(2.0, min(5.8, 0.085 * widest_line + 0.75))
        height = max(0.85, min(1.85, 0.42 * len(lines) + 0.38))

        if kind in {"start", "return"}:
            width = max(width, 1.8)
            height = max(height, 0.78)

        patch = FancyBboxPatch(
            (x - width / 2, y - height / 2),
            width,
            height,
            boxstyle="round,pad=0.10,rounding_size=0.16",
            linewidth=1.8,
            edgecolor=palette["edge"],
            facecolor=palette["face"],
            zorder=3,
        )

        ax.add_patch(patch)

        ax.text(
            x,
            y,
            label,
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold" if kind != "logpoint" else "normal",
            color=palette["text"],
            family="DejaVu Sans",
            zorder=4,
            wrap=True,
        )


def _draw_fsm_legend(ax) -> None:
    from matplotlib.patches import Patch

    handles = [
        Patch(
            facecolor="#059669",
            edgecolor="#064e3b",
            label="START",
        ),
        Patch(
            facecolor="#2563eb",
            edgecolor="#1e3a8a",
            label="LogPoint",
        ),
        Patch(
            facecolor="#166534",
            edgecolor="#14532d",
            label="RETURN",
        ),
        Patch(
            facecolor="#f59e0b",
            edgecolor="#92400e",
            label="Incomplete path",
        ),
    ]

    ax.legend(
        handles=handles,
        loc="upper right",
        frameon=True,
        framealpha=0.96,
        facecolor="#ffffff",
        edgecolor="#cbd5e1",
        fontsize=8,
    )


def _short_method_name(fullname: str) -> str:
    if not fullname:
        return "unknown"

    return fullname.rsplit(".", 1)[-1]


def _shorten_label(text: str, max_length: int) -> str:
    import textwrap

    normalized = " ".join((text or "").split())

    if len(normalized) > max_length:
        normalized = normalized[: max_length - 1].rstrip() + "…"

    return "\n".join(
        textwrap.wrap(
            normalized,
            width=24,
            break_long_words=False,
            break_on_hyphens=False,
        )
    )

In [11]:
%pip install graphviz

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os
import textwrap
import graphviz


def short_method(full_name: str) -> str:
    if not full_name:
        return ""
    return full_name.rsplit(".", 1)[-1] if "." in full_name else full_name


def shorten(s: str, width: int) -> str:
    s = (s or "").replace("\n", " ").strip()
    return textwrap.shorten(s, width=width, placeholder="...") if s else ""


def draw_branching_call_flow_graphviz(
    result,
    method_full_name: str,
    method_graph,
    filename: str = None,
    fmt: str = "png",
    max_condition_len: int = 28,
    max_paths_to_render: int = 200,
    rankdir: str = "LR",
    expand_links: bool = True,
    output_dir: str = "output",
    _rendered_registry: dict = None,
):
    """
    Строит структурный граф ветвления метода по method_paths.
    Узлы графа:
    - вызовы методов
    - RETURN
    - ромбы-ветвления по condition

    Если у вызванного метода есть собственные method_paths в result,
    для него создаётся отдельный файл.
    """
    _rendered_registry = _rendered_registry if _rendered_registry is not None else {}

    paths = result.method_paths.get(method_full_name, [])
    if not paths:
        raise ValueError(f"No paths for {method_full_name}")

    if len(paths) > max_paths_to_render:
        paths = paths[:max_paths_to_render]

    cs_map = {cs.node_id: cs for cs in getattr(method_graph, "call_sites", [])}

    def segment_calls(seg):
        calls = []
        seen = set()

        for cn in seg.nodes:
            if cn.label.upper() != "CALL":
                continue

            cs = cs_map.get(cn.node_id)
            callee_full = getattr(cs, "method_full_name", "") if cs else ""
            if not callee_full:
                continue

            display = short_method(callee_full)
            if not display or callee_full in seen:
                continue

            seen.add(callee_full)
            calls.append((callee_full, display))

        return tuple(calls)

    def extract_condition_text(prev_seg):
        if prev_seg is None or not prev_seg.nodes:
            return ""

        for cn in reversed(prev_seg.nodes):
            if cn.label.upper() == "CALL":
                code = (cn.code or "").strip()
                if code:
                    return shorten(code, max_condition_len)

        return ""

    def normalize_path(path):
        normalized = []
        prev_seg = None

        for seg in path.segments:
            calls = segment_calls(seg)
            cond = seg.condition or ""
            cond_text = extract_condition_text(prev_seg) if cond else ""
            is_terminal = bool(seg.is_terminal)

            if not calls and not cond and not is_terminal:
                prev_seg = seg
                continue

            if calls:
                for i, (callee_full, display) in enumerate(calls):
                    normalized.append({
                        "condition": cond if i == 0 else "",
                        "key": callee_full,
                        "display": display,
                        "is_terminal": False,
                        "condition_text": cond_text if i == 0 else "",
                        "callee_full": callee_full,
                    })

                if is_terminal:
                    normalized.append({
                        "condition": "",
                        "key": "__return__",
                        "display": "RETURN",
                        "is_terminal": True,
                        "condition_text": "",
                        "callee_full": None,
                    })
            else:
                normalized.append({
                    "condition": cond,
                    "key": "__return__" if is_terminal else "",
                    "display": "RETURN" if is_terminal else "",
                    "is_terminal": is_terminal,
                    "condition_text": cond_text,
                    "callee_full": None,
                })

            prev_seg = seg

        return normalized

    normalized_paths = []
    for path in paths:
        norm = normalize_path(path)
        if not norm:
            continue

        sigs = tuple(
            (
                item["condition"],
                item["is_terminal"],
                item["key"],
                item["condition_text"],
                item["display"],
                item["callee_full"],
            )
            for item in norm
        )
        normalized_paths.append(sigs)

    if not normalized_paths:
        raise ValueError(f"No normalized paths for {method_full_name}")

    class TrieNode:
        __slots__ = ("key", "children", "count", "terminal_count", "depth", "uid")
        _counter = [0]

        def __init__(self, key=None, depth=0):
            self.key = key
            self.children = {}
            self.count = 0
            self.terminal_count = 0
            self.depth = depth
            TrieNode._counter[0] += 1
            self.uid = f"n{TrieNode._counter[0]}"

    root = TrieNode(depth=0)

    for sigs in normalized_paths:
        cur = root
        cur.count += 1
        for sig in sigs:
            nxt = cur.children.get(sig)
            if nxt is None:
                nxt = TrieNode(key=sig, depth=cur.depth + 1)
                cur.children[sig] = nxt
            cur = nxt
            cur.count += 1
        cur.terminal_count += 1

    def cond_rank(sig):
        cond = sig[0]
        if cond in ("TRUE", "LOOP_TRUE"):
            return 0
        if cond in ("FALSE", "LOOP_FALSE"):
            return 1
        if cond == "LOOP_BODY":
            return 2
        return 3

    _ordered_cache = {}

    def ordered_children(node):
        cache_key = id(node)
        if cache_key in _ordered_cache:
            return _ordered_cache[cache_key]

        ordered = sorted(
            node.children.items(),
            key=lambda kv: (
                cond_rank(kv[0]),
                str(kv[0][2]),
                str(kv[0][3]),
                str(kv[0][1]),
            ),
        )
        _ordered_cache[cache_key] = ordered
        return ordered

    def box_text(sig):
        cond, is_terminal, key, cond_text, display, callee_full = sig
        if is_terminal:
            return "RETURN"
        if callee_full:
            return display
        return ""

    def should_draw_node(sig):
        cond, is_terminal, key, cond_text, display, callee_full = sig
        return bool(callee_full) or is_terminal

    def edge_label(sig):
        cond, is_terminal, key, cond_text, display, callee_full = sig
        if cond and cond_text:
            return f"{cond}\n{cond_text}"
        return cond or cond_text or None

    EDGE_COLOR = {
        "TRUE": "#22c55e",
        "LOOP_TRUE": "#22c55e",
        "FALSE": "#ef4444",
        "LOOP_FALSE": "#ef4444",
        "LOOP_BODY": "#f59e0b",
    }

    def edge_color(cond):
        return EDGE_COLOR.get(cond, "#94a3b8")

    g = graphviz.Digraph(
        "flow",
        graph_attr={
            "bgcolor": "#0f172a",
            "rankdir": rankdir,
            "splines": "spline",
            "nodesep": "0.35",
            "ranksep": "0.9",
            "fontname": "Helvetica",
            "label": f"{method_full_name}\nStructural call-flow graph",
            "labelloc": "t",
            "fontcolor": "white",
            "fontsize": "20",
        },
        node_attr={
            "fontname": "Helvetica",
            "fontsize": "11",
            "fontcolor": "white",
            "style": "rounded,filled",
            "penwidth": "1.3",
        },
        edge_attr={
            "fontname": "Helvetica",
            "fontsize": "9",
            "fontcolor": "white",
            "penwidth": "1.6",
        },
        format=fmt,
    )

    g.node(
        "root",
        label=method_graph.name,
        shape="box",
        style="rounded,filled",
        fillcolor="#0b3b2e",
        color="#34d399",
    )

    def merge_conditions(pending, new):
        p_label, p_cond = pending
        n_label, n_cond = new
        label = "\n".join(x for x in [p_label, n_label] if x) or None
        cond = p_cond or n_cond
        return (label, cond)

    def has_own_flow(callee_full):
        return bool(callee_full) and callee_full in result.method_paths

    def get_callee_method_graph(callee_full):
        for e in result.sequence:
            if e.method_graph.full_name == callee_full:
                return e.method_graph
        return None

    def visit(node, parent_uid, pending=(None, "")):
        children = ordered_children(node)
        if not children:
            return

        if len(children) > 1:
            diamond_uid = f"d_{node.uid}"
            p_label, p_cond = pending

            first_cond_text = next((sig[3] for sig, _ in children if sig[3]), "")
            diamond_label = p_label or first_cond_text or "?"

            g.node(
                diamond_uid,
                label=diamond_label,
                shape="diamond",
                fillcolor="#334155",
                color="#f8fafc",
                fontsize="9",
            )
            g.edge(
                parent_uid,
                diamond_uid,
                label=p_cond if p_cond else None,
                color=edge_color(p_cond) if p_cond else "#cbd5e1",
            )
            branch_from = diamond_uid
            pending = (None, "")
        else:
            branch_from = parent_uid

        for sig, child in children:
            cond, is_terminal, key, cond_text, display, callee_full = sig
            this_label = edge_label(sig)
            combined = merge_conditions(pending, (this_label, cond))

            if should_draw_node(sig):
                if is_terminal:
                    fillcolor, color = "#3f1d2e", "#f472b6"
                    shape = "box"
                else:
                    fillcolor, color = "#132a3a", "#38bdf8"
                    shape = "box"

                node_label = box_text(sig)
                child_has_flow = has_own_flow(callee_full)
                if child_has_flow:
                    node_label = f"{node_label}\n[expand]"
                    color = "#a78bfa"

                g.node(
                    child.uid,
                    label=node_label,
                    shape=shape,
                    style="rounded,filled",
                    fillcolor=fillcolor,
                    color=color,
                )

                _, cond_c = combined
                g.edge(
                    branch_from,
                    child.uid,
                    label=cond_c or None,
                    color=edge_color(cond_c),
                )

                if child_has_flow and expand_links and callee_full not in _rendered_registry:
                    _rendered_registry[callee_full] = True
                    callee_mg = get_callee_method_graph(callee_full)

                    if callee_mg is not None:
                        safe_name = "".join(
                            c if c.isalnum() else "_" for c in short_method(callee_full)
                        ) or "subgraph"

                        try:
                            draw_branching_call_flow_graphviz(
                                result=result,
                                method_full_name=callee_full,
                                method_graph=callee_mg,
                                filename=f"sub_{safe_name}",
                                fmt=fmt,
                                max_condition_len=max_condition_len,
                                max_paths_to_render=max_paths_to_render,
                                rankdir=rankdir,
                                expand_links=expand_links,
                                output_dir=output_dir,
                                _rendered_registry=_rendered_registry,
                            )
                        except ValueError:
                            pass

                visit(child, child.uid)
            else:
                visit(child, branch_from, pending=combined)

    visit(root, "root")

    os.makedirs(output_dir, exist_ok=True)
    render_name = filename or short_method(method_full_name) or "branching_graph"
    out_path = g.render(filename=render_name, directory=output_dir, cleanup=True)
    return out_path

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

from cpg.entrypoint import EntrypointDetector
from cpg.flow import EntrypointFlow
from cpg.log_flow import LogFlowExtractor


OUTPUT_DIR = Path(f"output/{SERVICE_NAME}/fsms")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_DEPTH = 10
MAX_LOOP_UNROLL = 1
MAX_PATHS = 256

extractor = LogFlowExtractor(templates)
flow_analyzer = EntrypointFlow(G, max_depth=5, max_paths=50)

def safe_filename(value: str, max_length: int = 100) -> str:
    value = value.strip()
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value)
    value = value.strip("._-")

    if not value:
        value = "unnamed"

    return value[:max_length]


def has_start_transition(fsm) -> bool:
    return any(edge.source == "__start__" for edge in fsm.edges)


rows = []
fsms = {}

for index, entrypoint in enumerate(all_entrypoints, start=1):
    entrypoint_id = entrypoint.node_id
    entrypoint_name = entrypoint.name
    entrypoint_fullname = entrypoint.full_name

    base_name = (
        f"{index:03d}_"
        f"{safe_filename(entrypoint_name)}_"
        f"{safe_filename(entrypoint_fullname)}"
    )

    png_path = OUTPUT_DIR / f"{base_name}.png"

    row = {
        "rank": index,
        "entrypoint_nodeid": entrypoint_id,
        "entrypoint_name": entrypoint_name,
        "entrypoint_fullname": entrypoint_fullname,
        "outdegree": getattr(entrypoint, "outdegree", ""),
        "status": "ok",
        "states": 0,
        "edges": 0,
        "terminals": 0,
        "has_start": False,
        "png": str(png_path),
        "warnings": "",
        "error": "",
    }

    
    flow_result = flow_analyzer.build(entrypoint.node_id)
    mg = next((e.method_graph for e in flow_result.sequence if e.method_graph.full_name == entrypoint.full_name), None)
    output_path = draw_branching_call_flow_graphviz(
        flow_result,
        method_full_name=entrypoint.full_name,
        method_graph=mg,
        filename="handler",
        output_dir=f"./output/{SERVICE_NAME}/flow/"
    )
    print(output_path)

    fsm = extractor.extract(flow_result)
    fsms[entrypoint.full_name] = fsm

    if fsm.states and fsm.edges:
        visualize_log_fsm(
            fsm,
            output_path=str(png_path),
        )
    else:
        row["status"] = "no_log_fsm"

    row["states"] = len(fsm.states)
    row["edges"] = len(fsm.edges)
    row["terminals"] = len(fsm.terminals)
    row["has_start"] = has_start_transition(fsm)
    row["warnings"] = " | ".join(fsm.warnings)

    rows.append(row)

    print(
        f"[{index:03d}/{len(all_entrypoints):03d}] "
        f"{entrypoint_name}: "
        f"{row['status']}, "
        f"states={row['states']}, "
        f"edges={row['edges']}"
    )

output\frontend\flow\handler.png
[001/062] init: no_log_fsm, states=0, edges=0
output\frontend\flow\handler.png
[002/062] Descriptor: no_log_fsm, states=0, edges=0
output\frontend\flow\handler.png
[003/062] Descriptor: no_log_fsm, states=0, edges=0
output\frontend\flow\handler.png
[004/062] Descriptor: no_log_fsm, states=0, edges=0
output\frontend\flow\handler.png
[005/062] Descriptor: no_log_fsm, states=0, edges=0
output\frontend\flow\handler.png
[006/062] Descriptor: no_log_fsm, states=0, edges=0
output\frontend\flow\handler.png
[007/062] Descriptor: no_log_fsm, states=0, edges=0
output\frontend\flow\handler.png
[008/062] Descriptor: no_log_fsm, states=0, edges=0
output\frontend\flow\handler.png
[009/062] Descriptor: no_log_fsm, states=0, edges=0
output\frontend\flow\handler.png
[010/062] Descriptor: no_log_fsm, states=0, edges=0
output\frontend\flow\handler.png
[011/062] Descriptor: no_log_fsm, states=0, edges=0
output\frontend\flow\handler.png
[012/062] Descriptor: no_log_fsm, stat

In [14]:
print("templates:", len(extractor.templates_by_call))
print("fsm states:", len(fsm.states))
print("fsm external states:", len(fsm.external_states))
print("fsm edges:", len(fsm.edges))
print("fsm terminals:", fsm.terminals)

templates: 77
fsm states: 0
fsm external states: 0
fsm edges: 0
fsm terminals: set()


In [21]:
from typing import List, Tuple
import csv

def extract_logs(dataset_path: str, service: str, start_time: int, end_time: int) -> List[Tuple[str, str]]:
    logs = []
    try:
        with open(dataset_path, 'r') as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if len(row) < 3:
                    continue
                try:
                    timestamp   = int(float(row[0]))
                    log_service = row[1].strip()
                    message     = row[2].strip()
                    if log_service.lower() == service.lower() and start_time <= timestamp <= end_time:
                        bucket = f"{log_service}@{timestamp}"
                        logs.append((bucket, message))
                except (ValueError, IndexError):
                    continue
    except FileNotFoundError:
        print(f"ERROR: Dataset file not found: {dataset_path}")
    print(f"✓ Extracted {len(logs)} logs for {service} in time window")
    return logs

In [22]:
dataset = "./dataset/re3ob_adservice_f3_1/logs.csv"
start = 1731903974
end = 1731903980
logs = extract_logs(dataset, SERVICE_NAME, start, end)

✓ Extracted 107 logs for frontend in time window


In [23]:
import pandas as pd
from collections import defaultdict

# ── Шаг 1: маппинг логов с добавлением full_name ──────────────────────────
mappings = cpg_parser.map_logs(logs, root, templates, min_static=1)

# Быстрый lookup: method_node_id → full_name
node_to_fullname = {
    nid: data.get("FULL_NAME", "")
    for nid, data in G.nodes(data=True)
    if data.get("label") == '"METHOD"'
}

rows = []
for m in mappings:
    full_name = node_to_fullname.get(m.method_node_id, "—") if m.method_node_id else "—"
    rows.append({
        "bucket":         m.bucket,
        "matched":        "✓" if m.matched else "✗",
        "method_name":    m.method_name    if m.method_name    else "—",
        "full_name":      full_name,                              # ← новое поле
        "template":       m.template       if m.template       else "—",
        "score":          m.score,
        "call_node_id":   m.call_node_id   if m.call_node_id   else "—",
        "method_node_id": m.method_node_id if m.method_node_id else "—",
        "message":        m.message,
    })

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)
display(df)

,bucket,matched,method_name,full_name,template,score,call_node_id,method_node_id,message
0,frontend@1731903974,✓,ServeHTTP,"""main.logHandler.ServeHTTP""",request started <*>,2,30064774247,107374182800,request started
1,frontend@1731903974,✓,productHandler,"""main.frontendServer.productHandler""",serving product page <*>,3,30064773422,107374182767,serving product page
2,frontend@1731903974,✗,—,—,—,0,—,—,request complete
3,frontend@1731903974,✗,—,—,—,0,—,—,request complete
4,frontend@1731903974,✓,ServeHTTP,"""main.logHandler.ServeHTTP""",request started <*>,2,30064774247,107374182800,request started
5,frontend@1731903974,✓,addToCartHandler,"""main.frontendServer.addToCartHandler""",adding to cart <*>,3,30064773495,107374182768,adding to cart
6,frontend@1731903974,✗,—,—,—,0,—,—,request complete
7,frontend@1731903974,✓,ServeHTTP,"""main.logHandler.ServeHTTP""",request started <*>,2,30064774247,107374182800,request started
8,frontend@1731903974,✓,viewCartHandler,"""main.frontendServer.viewCartHandler""",view user cart <*>,3,30064773548,107374182770,view user cart
9,frontend@1731903974,✗,—,—,—,0,—,—,request complete


In [ ]:
from cpg.log_chain_classifier import run_multi_fsm_analyzer

result = run_multi_fsm_analyzer(
    G=G,
    templates=templates,
    all_entrypoints=all_entrypoints,
    service_name=SERVICE_NAME,
    dataset_path="./dataset/re3ob_adservice_f3_1/logs.csv",
    start_time=1731903974,
    end_time=1731903980,
)